# Santander Customer Transaction Prediction - Full Project Notebook

This notebook is designed for **EC452 Semester Project** and provides a complete, reproducible pipeline for the Kaggle competition: Santander Customer Transaction Prediction.

**Competition**: https://www.kaggle.com/competitions/santander-customer-transaction-prediction

---
## Notebook Contents
1. Setup and Imports
2. Data Loading
3. EDA (distribution, imbalance, correlations, sanity checks)
4. Baseline Model (Logistic Regression + StandardScaler)
5. Advanced Models (LightGBM / XGBoost / CatBoost fallback)
6. Feature Selection Experiments
7. Hyperparameter Tuning
8. Explainability (SHAP + Permutation Importance + PDP)
9. Final Training & Kaggle Submission
10. Reproducibility Appendix


In [ ]:
import os
import gc
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Optional model libraries
HAVE_LGBM = HAVE_XGB = HAVE_CAT = False
try:
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except Exception:
    pass

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception:
    pass

try:
    from catboost import CatBoostClassifier
    HAVE_CAT = True
except Exception:
    pass

print(f'LightGBM available: {HAVE_LGBM}')
print(f'XGBoost available: {HAVE_XGB}')
print(f'CatBoost available: {HAVE_CAT}')

In [ ]:
# Data paths for Kaggle environment
TRAIN_PATH = '/kaggle/input/santander-customer-transaction-prediction/train.csv'
TEST_PATH = '/kaggle/input/santander-customer-transaction-prediction/test.csv'
SUB_PATH = '/kaggle/input/santander-customer-transaction-prediction/sample_submission.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sub_df = pd.read_csv(SUB_PATH)

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
train_df.head()

In [ ]:
# Basic checks
print('Missing values in train:', train_df.isna().sum().sum())
print('Missing values in test :', test_df.isna().sum().sum())

target_rate = train_df['target'].mean()
print(f'Positive class ratio: {target_rate:.4f}')

feature_cols = [c for c in train_df.columns if c.startswith('var_')]
print('Number of features:', len(feature_cols))

In [ ]:
# EDA: target distribution
plt.figure(figsize=(6,4))
sns.countplot(x='target', data=train_df)
plt.title('Target Distribution')
plt.show()

In [ ]:
# EDA: random feature distributions
sample_features = np.random.choice(feature_cols, size=9, replace=False)
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, col in zip(axes.ravel(), sample_features):
    sns.kdeplot(data=train_df, x=col, hue='target', common_norm=False, fill=True, alpha=0.3, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# EDA: correlation heatmap for top-variance features
top_var_features = train_df[feature_cols].var().sort_values(ascending=False).head(30).index.tolist()
corr = train_df[top_var_features].corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap (Top 30 Variance Features)')
plt.show()

## Baseline Model
As required: **Logistic Regression + StandardScaler** on full feature set, evaluated with ROC-AUC using stratified cross-validation.

In [ ]:
X = train_df[feature_cols].copy()
y = train_df['target'].copy()
X_test = test_df[feature_cols].copy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, random_state=SEED))
])

baseline_auc = cross_val_score(baseline, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)
print('Baseline ROC-AUC folds:', np.round(baseline_auc, 5))
print('Baseline ROC-AUC mean :', baseline_auc.mean())

## Advanced Models

In [ ]:
model_results = {}

if HAVE_LGBM:
    lgbm = LGBMClassifier(
        n_estimators=600,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary',
        random_state=SEED
    )
    auc = cross_val_score(lgbm, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)
    model_results['LightGBM'] = auc

if HAVE_XGB:
    xgb = XGBClassifier(
        n_estimators=700,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='auc',
        random_state=SEED
    )
    auc = cross_val_score(xgb, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)
    model_results['XGBoost'] = auc

if HAVE_CAT:
    cat = CatBoostClassifier(
        iterations=900,
        learning_rate=0.03,
        depth=6,
        loss_function='Logloss',
        eval_metric='AUC',
        verbose=0,
        random_seed=SEED
    )
    auc = cross_val_score(cat, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)
    model_results['CatBoost'] = auc

for name, aucs in model_results.items():
    print(f'{name} ROC-AUC mean: {aucs.mean():.6f} | folds: {np.round(aucs,5)}')

In [ ]:
# Feature selection experiments (for linear model)
selection_pipelines = {
    'VarianceThreshold + LR': Pipeline([
        ('var', VarianceThreshold(0.0)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=SEED))
    ]),
    'SelectKBest(80) + LR': Pipeline([
        ('select', SelectKBest(score_func=f_classif, k=80)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=SEED))
    ]),
    'SelectKBest(120) + LR': Pipeline([
        ('select', SelectKBest(score_func=f_classif, k=120)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=SEED))
    ])
}

for name, pipe in selection_pipelines.items():
    auc = cross_val_score(pipe, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)
    print(f'{name}: {auc.mean():.6f}')

In [ ]:
# Select final model by best CV score
candidate_models = {'Baseline_LR': baseline_auc.mean()}
for name, aucs in model_results.items():
    candidate_models[name] = aucs.mean()

best_model_name = max(candidate_models, key=candidate_models.get)
print('Best model:', best_model_name, '| ROC-AUC:', candidate_models[best_model_name])

## Explainability (SHAP + Permutation Importance + PDP)

In [ ]:
# Fit interpretable reference model (Logistic Regression) for feature-level analysis
baseline.fit(X, y)

perm = permutation_importance(baseline, X, y, n_repeats=5, scoring='roc_auc', random_state=SEED, n_jobs=-1)
perm_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(data=perm_importance.head(20), x='importance', y='feature')
plt.title('Top 20 Permutation Importances (Baseline LR)')
plt.show()

In [ ]:
# PDP for top features
top_feats = perm_importance.head(4)['feature'].tolist()
fig, ax = plt.subplots(2, 2, figsize=(12,8))
PartialDependenceDisplay.from_estimator(baseline, X, features=top_feats, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP analysis
try:
    import shap
    sample_idx = np.random.choice(np.arange(len(X)), size=min(5000, len(X)), replace=False)
    X_sample = X.iloc[sample_idx]

    # Use model-specific explainer where possible
    if best_model_name == 'LightGBM' and HAVE_LGBM:
        final_for_shap = LGBMClassifier(n_estimators=600, learning_rate=0.03, random_state=SEED)
        final_for_shap.fit(X, y)
        explainer = shap.TreeExplainer(final_for_shap)
        shap_values = explainer.shap_values(X_sample)
    else:
        # Fallback: linear/logistic model explanation
        explainer = shap.Explainer(baseline.named_steps['clf'], baseline.named_steps['scaler'].transform(X_sample))
        shap_values = explainer(baseline.named_steps['scaler'].transform(X_sample))

    print('SHAP computed successfully.')
except Exception as e:
    print('SHAP could not run in current environment:', e)

## Final Training and Submission

In [ ]:
# Train final model on full train and generate test probabilities
if best_model_name == 'LightGBM' and HAVE_LGBM:
    final_model = LGBMClassifier(
        n_estimators=800, learning_rate=0.025, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED
    )
    final_model.fit(X, y)
    test_pred = final_model.predict_proba(X_test)[:, 1]
elif best_model_name == 'XGBoost' and HAVE_XGB:
    final_model = XGBClassifier(
        n_estimators=900, max_depth=4, learning_rate=0.025,
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc', random_state=SEED
    )
    final_model.fit(X, y)
    test_pred = final_model.predict_proba(X_test)[:, 1]
else:
    final_model = baseline
    final_model.fit(X, y)
    test_pred = final_model.predict_proba(X_test)[:, 1]

sub_df['target'] = test_pred
sub_df.to_csv('submission.csv', index=False)
print('Saved: submission.csv')
sub_df.head()

## Reproducibility Appendix
- Python: designed for Kaggle notebook environment
- Random seed: 42
- CV: Stratified 5-fold
- Metric: ROC-AUC
- Baseline: StandardScaler + LogisticRegression
- Advanced models: LightGBM / XGBoost / CatBoost (if available)
- Explainability: SHAP + Permutation Importance + PDP